# Project Lifecycle: Conversational Movie Recommendation System
**Notebook 2: Representation Learning & Candidate Retrieval**


##Problem Statement
In a production recommendation system, we cannot afford to run a complex ranking model on thousands of items for every user query. Instead, we use a Two-Stage Architecture:

1. Retrieval (This Notebook): Efficiently narrow down the entire catalog to a high-recall "Candidate Pool" of 50–100 items.

2. Ranking (Notebook 3): Apply a high-precision model to sort those few candidates.

This notebook implements the Stage 1 Retrieval pipeline by:

* Implementing TF-IDF as a lexical baseline for exact keyword matching.

* Leveraging Sentence Transformers (Bi-Encoders) to capture deep semantic meaning.

* Integrating FAISS (Facebook AI Similarity Search) for sub-millisecond production-scale retrieval.

* Using Quantitative Metrics (Diversity Score) and Failure Mode Analysis to validate the system before moving to the ranking stage.

### Imports & Data Loading

In [16]:
import pandas as pd
import numpy as np
import time
import faiss
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import plotly.express as px

# Load the high-signal dataset from Notebook 1
df = pd.read_parquet('movies_processed_final.parquet')
print(f"Loaded {len(df)} movies ready for vectorization.")

Loaded 27264 movies ready for vectorization.


## 1. Lexical Baseline (TF-IDF)
TF-IDF (Term Frequency-Inverse Document Frequency) is our baseline. It is excellent for exact matches (names, specific genres) but lacks an understanding of synonyms.

### 1.1 TF-IDF Implementation

In [6]:
# Initialize TF-IDF with a limit on features to reduce noise
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['soup'])

def get_tfidf_recommendations(query, top_n=10):
    query_vec = tfidf.transform([query.lower()])
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()

    indices = similarity.argsort()[-top_n:][::-1]
    results = df.iloc[indices].copy()
    results['score'] = similarity[indices]
    return results[['title', 'year', 'score']]

# Test Case: Exact keyword matching
get_tfidf_recommendations("Christopher Nolan Batman", top_n=5)

,title,year,score
16003,Christopher Strong (1933),2000.0,0.468227
16441,Batman: Year One (2011),2000.0,0.439689
22579,Batman: Mystery of the Batwoman (2003),2000.0,0.432056
16308,Batman (1943),2000.0,0.408979
23373,Son of Batman (2014),2000.0,0.401142


## 2. Semantic Engine (Sentence Transformers & FAISS)
To handle abstract queries like "mind-bending thrillers," we use a Bi-Encoder to map movies into a 384-dimensional vector space. To ensure this scales to millions of movies, we use FAISS for retrieval.

### 2.1 Embedding & Indexing

In [7]:
# 'all-MiniLM-L6-v2' is optimized for a balance of speed and semantic accuracy
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings...")
embeddings = model.encode(df['soup'].tolist(), show_progress_bar=True, convert_to_numpy=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings...


Batches:   0%|          | 0/852 [00:00<?, ?it/s]

In [8]:
# 1. Prepare embeddings for FAISS
# FAISS requires float32 and C-contiguous memory layout
embeddings_f32 = np.ascontiguousarray(embeddings.astype('float32'))

# 2. Initialize the Index
dimension = embeddings_f32.shape[1]
# IndexFlatIP = Inner Product. When vectors are normalized, this equals Cosine Similarity.
index = faiss.IndexFlatIP(dimension)

# 3. Normalize and Add to Index
faiss.normalize_L2(embeddings_f32)
index.add(embeddings_f32)

print(f"Success: Index built with {index.ntotal} vectors.")

def get_semantic_recommendations(query, top_n=10):
    # a. Encode the query
    query_vec = model.encode([query.lower()]).astype('float32')

    # b. Ensure the query vector is C-contiguous for FAISS
    query_vec = np.ascontiguousarray(query_vec)

    # c. Normalize the query vector
    faiss.normalize_L2(query_vec)

    # d. Search the index
    # distances = similarity scores, indices = row positions in df
    distances, indices = index.search(query_vec, top_n)

    # e. Extract results
    results = df.iloc[indices[0]].copy()
    results['score'] = distances[0] # FAISS returns scores in distances

    return results[['title', 'year', 'score']]

# Test
get_semantic_recommendations("Inception", top_n=5)

Success: Index built with 27264 vectors.


,title,year,score
0,Inception (2010),2010.0,0.614901
27103,Into the Deep (1994),2000.0,0.469652
19093,Dark Circles (2013),2000.0,0.456551
24924,Neurons to Nirvana (2013),2000.0,0.455893
23793,"Initiation, The (1984)",2000.0,0.451361


## 3.Evaluation
### 3.1 Qualitative Comparison
We compare both strategies side-by-side to understand how the "Retrieval Surface" changes between keyword matching and semantic understanding

In [9]:
def compare_strategies(query, top_n=5):
    print(f" Query: '{query}'\n" + "-"*50)
    lexical = get_tfidf_recommendations(query, top_n)
    semantic = get_semantic_recommendations(query, top_n)

    comparison = pd.DataFrame({
        "Rank": range(1, top_n + 1),
        "TF-IDF (Lexical)": lexical['title'].values,
        "Semantic (Meaning)": semantic['title'].values
    })
    return comparison

compare_strategies("movies about deep sea exploration and monsters")

 Query: 'movies about deep sea exploration and monsters'
--------------------------------------------------


,Rank,TF-IDF (Lexical),Semantic (Meaning)
0,1,"Deep Blue Sea, The (2011)",Deep Sea 3D (2006)
1,2,Deep Sea 3D (2006),Deep Rising (1998)
2,3,Between the Devil and the Deep Blue Sea (1995),"Terror Beneath the Sea, The (Kaitei daisensô) ..."
3,4,For the Love of Movies (2009),Black Sea (2014)
4,5,Four Eyed Monsters (2005),Sharktopus (2010)


### Insight: TF-IDF vs Semantic Retrieval

* For vague queries like "deep sea exploration and monsters", TF-IDF mostly returned movies with overlapping keywords such as "deep" or "sea", even if they were not truly relevant.

* In contrast, the embedding-based approach captured the underlying theme (ocean + danger + creatures) and returned more contextually appropriate movies.

* This highlights a key limitation of lexical methods and shows why semantic embeddings are better suited for natural language queries.

While embeddings improve retrieval quality, the system still lacks ranking and personalization, which will be addressed in the hybrid model.

In [19]:
compare_strategies("mind bending psychological movies")

 Query: 'mind bending psychological movies'
--------------------------------------------------


,Rank,TF-IDF (Lexical),Semantic (Meaning)
0,1,For the Love of Movies (2009),Into the Mind (2013)
1,2,Into the Mind (2013),Free The Mind (2012)
2,3,"Men Who Made the Movies: Samuel Fuller, The (2...",Like Minds (Murderous Intent) (2006)
3,4,A Night at the Movies: The Suspenseful World o...,Mindwarp (1992)
4,5,Midnight Movies: From the Margin to the Mainst...,Mind Game (2004)


In [11]:
compare_strategies("light hearted romantic comedy")

 Query: 'light hearted romantic comedy'
--------------------------------------------------


,Rank,TF-IDF (Lexical),Semantic (Meaning)
0,1,Romantic Comedy (1983),"Romantics, The (2010)"
1,2,"Romantic Englishwoman, The (1975)",Romantic Comedy (1983)
2,3,Light Gradient (2009),Funny About Love (1990)
3,4,Light of Day (1987),My Girlfriend's Boyfriend (2010)
4,5,You Light Up My Life (1977),Shirin in Love (2014)


###  Observations from Query Experiments

* For vague and abstract queries like "mind bending psychological movies" and "light hearted romantic comedy", TF-IDF produced results driven by literal keyword matches rather than actual intent.

* For example, it matched words like "light" or "movies", leading to irrelevant recommendations.

* In contrast, the embedding-based approach captured the underlying meaning of the query, returning movies that align better with themes such as psychological depth or romantic tone.

* However, some results are still loosely relevant, showing that retrieval alone is not sufficient and requires a ranking layer for refinement.

### 3.2 Failure Mode Analysis
Failure Mode: Logical Constraints

Semantic search captures meaning but struggles with logical constraints like "NOT".

This highlights the limitation of pure vector search.


In [20]:
query = "movies directed by Christopher Nolan but not about space"
results = get_semantic_recommendations(query, top_n=5)

print(f"Query: {query}")
results

Query: movies directed by Christopher Nolan but not about space


,title,year,score
24226,"Space Movie, The (1980)",2000.0,0.565771
12037,First Man Into Space (1959),2000.0,0.551621
11718,Plan 9 from Outer Space (1959),2000.0,0.549064
26150,Space (1985),2000.0,0.546728
12519,Space Is The Place (1974),2000.0,0.535529


In [21]:
query = "romantic movies but not comedy"
results = get_semantic_recommendations(query, top_n=5)

print(f"Query: {query}")
results

Query: romantic movies but not comedy


,title,year,score
16051,"Romantics, The (2010)",2000.0,0.745065
5807,Romantic Comedy (1983),2000.0,0.701991
16892,My Girlfriend's Boyfriend (2010),2000.0,0.690088
25343,1987 (2014),2000.0,0.681910
5045,Born Romantic (2000),2000.0,0.680424


### Failure Mode Analysis: Structured Constraints & Query Logic

To evaluate the robustness of the retrieval system, we tested queries containing
both semantic intent and logical constraints.

#### Test Query
"movies directed by Christopher Nolan but not about space"

### Observation
The model retrieves only space-related movies and completely ignores the director constraint.

### Root Causes

1. Missing Structured Features  
   The current text representation ("soup") does not include director information,
   so the model has no signal to associate movies with Christopher Nolan.

2. Dominant Semantic Signal  
   The term "space" dominates the embedding, leading the model to prioritize
   space-related movies over other aspects of the query.

3. Lack of Logical Reasoning  
   Embedding-based retrieval captures similarity but does not support logical
   operators such as "NOT" or conditional constraints.

### Takeaway

This experiment highlights that:

- The retrieval stage is optimized for **semantic recall**, not logical precision  
- Structured constraints (e.g., director, year) cannot be reliably handled using text similarity alone  
- A production system should combine:
  - Semantic retrieval (for candidate generation)
  - Structured filtering or ranking (for constraint enforcement)

### 3.3 Quantitative Evaluation (Diversity Score)
#### Diversity vs Relevance Trade-off

We use the Diversity Score to measure the "cohesion" of our results. A lower diversity in semantic search often indicates the model has successfully identified a tight "thematic neighborhood."

In [12]:
def compute_diversity(indices, embeddings_matrix):
    # Subset embeddings for the recommended items
    vectors = embeddings_matrix[indices]
    sim_matrix = cosine_similarity(vectors)

    # Calculate average similarity between all pairs (excluding self)
    upper_triangle = sim_matrix[np.triu_indices(len(indices), k=1)]
    return 1 - np.mean(upper_triangle)

# Example: Testing 'Inception'
# 1. Get Indices for top 10 results
semantic_idx = get_semantic_recommendations("Inception", top_n=10).index
tfidf_idx = get_tfidf_recommendations("Inception", top_n=10).index

print(f"TF-IDF Diversity: {compute_diversity(tfidf_idx, embeddings):.4f}")
print(f"Semantic Diversity: {compute_diversity(semantic_idx, embeddings):.4f}")

TF-IDF Diversity: 0.8533
Semantic Diversity: 0.5533


## Diversity vs Relevance Trade-off

TF-IDF produced a higher diversity score (~0.853), but this came at the cost of relevance, often returning loosely related or keyword-matching results.

Embeddings showed a more balanced behavior (~0.553), where recommendations were semantically similar while still maintaining variety.

This highlights an important trade-off: higher diversity does not always mean better recommendations — relevance must be preserved.

### 3.4 Exploratory Visualization (t-SNE)
We project our 384D embeddings into 2D space to verify that our movies are clustering into logical thematic neighborhoods.

In [18]:
def visualize_clusters(n_samples=500):
    sub_df = df.sample(n_samples, random_state=42)
    sub_embeddings = model.encode(sub_df['soup'].tolist())

    tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
    embeddings_2d = tsne.fit_transform(sub_embeddings)

    sub_df['x'], sub_df['y'] = embeddings_2d[:, 0], embeddings_2d[:, 1]

    fig = px.scatter(sub_df, x='x', y='y', hover_data=['title', 'year'],
                     color='vote_average', title="Movie Embedding Clusters (t-SNE Projection)",
                     template="plotly_dark")
    fig.show()

visualize_clusters()

### Final Retrieval Strategy

Based on the experiments:

- TF-IDF shows higher diversity but lacks semantic understanding
- Embeddings capture intent and produce more relevant recommendations
- Diversity remains acceptable without degrading relevance

Therefore, **embedding-based retrieval is selected as the primary method**.

TF-IDF is retained as a baseline and fallback option.

### Role in the Recommendation System

This retrieval module serves as the **candidate generation layer**.

It retrieves a set of relevant movies, which are then passed to a
ranking model in the next stage for final ordering.

In [14]:
import pickle

# Save FAISS index and movie metadata
faiss.write_index(index, "movies_faiss.index")
df.to_parquet("movies_with_embeddings.parquet")
# Save the TF-IDF vectorizer and matrix
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
# Save the model name (or the whole model if small)
with open("model_name.txt", "w") as f:
    f.write("all-MiniLM-L6-v2")

print("Artifacts saved for Notebook 3.")

Artifacts saved for Notebook 3.
